# Confusion Matrix & Evaluation Metrics
**Part A:** Numerical question (200 patients) solved by hand + code  
**Part B:** Same workflow on a real Kaggle dataset (preprocessing -> model -> metrics -> visuals)

## Part A: Numerical Question

**Given:** 200 patients, 80 actually diseased, model catches 70, and 20 healthy patients are wrongly flagged.

- TP = 70
- FN = 80 - 70 = 10
- FP = 20
- TN = 200 - (70 + 10 + 20) = 100

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

TP, FN, FP = 70, 10, 20
TN = 200 - (TP + FN + FP)

cm = np.array([[TN, FP],
               [FN, TP]])   # rows = actual (No, Yes), cols = predicted (No, Yes)

accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)

print(f"TP={TP}  TN={TN}  FP={FP}  FN={FN}")
print(f"Accuracy  = {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"Precision = {precision:.4f}  ({precision*100:.1f}%)")
print(f"Recall    = {recall:.4f}  ({recall*100:.1f}%)")
print(f"F1-Score  = {f1:.4f}  ({f1*100:.1f}%)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[0],
            xticklabels=["Predicted: No", "Predicted: Yes"],
            yticklabels=["Actual: No", "Actual: Yes"])
ax[0].set_title("Confusion Matrix (200 patients)")

names = ["Accuracy", "Precision", "Recall", "F1-Score"]
vals = [accuracy, precision, recall, f1]
bars = ax[1].bar(names, vals, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
ax[1].set_ylim(0, 1.05)
ax[1].set_title("Evaluation Metrics")
for b, v in zip(bars, vals):
    ax[1].text(b.get_x() + b.get_width()/2, v + 0.015, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.savefig("part_a_results.png", dpi=150)
plt.show()

### Hand-calculation summary
| Metric | Formula | Value |
|---|---|---|
| Accuracy | (70+100)/200 | 0.85 |
| Precision | 70/(70+20) | 0.778 |
| Recall | 70/(70+10) | 0.875 |
| F1-Score | 2PR/(P+R) = 140/170 | 0.824 |

## Part B: Real dataset (Kaggle) on Google Colab
Uses the **Pima Indians Diabetes** dataset (`uciml/pima-indians-diabetes-database`). If the Kaggle download fails, it falls back to scikit-learn's built-in breast cancer dataset so the notebook always runs.

In [ ]:
!pip -q install kagglehub

In [ ]:
import pandas as pd

try:
    import kagglehub, os
    path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")
    df = pd.read_csv(os.path.join(path, "diabetes.csv"))
    target = "Outcome"
    print("Loaded Kaggle dataset")
except Exception as e:
    print("Kaggle download failed, using sklearn fallback:", e)
    from sklearn.datasets import load_breast_cancer
    d = load_breast_cancer(as_frame=True)
    df = d.frame
    target = "target"

print(df.shape)
df.head()

### Data preprocessing

In [ ]:
print(df.isnull().sum().sum(), "missing values (NaN)")
print(df.duplicated().sum(), "duplicate rows")

# In the Pima dataset, 0 is an impossible value for these columns -> treat as missing
zero_invalid = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for c in zero_invalid:
    if c in df.columns:
        df[c] = df[c].replace(0, np.nan)

df = df.drop_duplicates()
df = df.fillna(df.median(numeric_only=True))     # impute with median

df[target].value_counts().plot(kind="bar", title="Class balance")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, classification_report,
                             ConfusionMatrixDisplay, roc_curve, auc)

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

cm2 = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm2.ravel()
print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")
print(f"Accuracy  = {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision = {precision_score(y_test, y_pred):.4f}")
print(f"Recall    = {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score  = {f1_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

ConfusionMatrixDisplay(cm2, display_labels=["No", "Yes"]).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title("Confusion Matrix (test set)")

m = [accuracy_score(y_test, y_pred), precision_score(y_test, y_pred),
     recall_score(y_test, y_pred), f1_score(y_test, y_pred)]
bars = ax[1].bar(names, m, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
ax[1].set_ylim(0, 1.05); ax[1].set_title("Evaluation Metrics")
for b, v in zip(bars, m):
    ax[1].text(b.get_x() + b.get_width()/2, v + 0.015, f"{v:.3f}", ha="center")

fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test_s)[:, 1])
ax[2].plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.3f}")
ax[2].plot([0, 1], [0, 1], "--", color="gray")
ax[2].set_title("ROC Curve"); ax[2].set_xlabel("FPR"); ax[2].set_ylabel("TPR"); ax[2].legend()

plt.tight_layout()
plt.savefig("part_b_results.png", dpi=150)
plt.show()

## Push to GitHub
1. In Colab: **File -> Save a copy in GitHub**, or download the `.ipynb` and upload it to a new repo.
2. Also upload `part_a_results.png` and `part_b_results.png` (Colab left sidebar -> Files).
3. Share the repo link with your results.